<a href="https://colab.research.google.com/github/DanylchenkoKateryna/NLP-Lab-works/blob/main/notebooks/lab11_llm_extraction_schema_first.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Lab 11 — LLM Extraction as Engineering (schema-first)

**Corpus**: 20 Newsgroups (alt.atheism, sci.electronics, soc.religion.christian)  
**Task**: Structured JSON extraction + JSON schema validation + repair loop  
**Key metric**: Valid JSON rate (before and after repair)

## 1. Install Dependencies

In [18]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jsonschema"], check=True)
    print("Colab: jsonschema installed.")
else:
    print("Local environment -- dependencies assumed installed.")

Colab: jsonschema installed.


## 2. Data Access

Clone repo in Colab (`src/` files already in repo), add `src/` to sys.path.

In [19]:
import os, sys, warnings, json
import warnings
warnings.filterwarnings('ignore')

In [20]:
from pathlib import Path
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("/content/NLP-Lab-works").exists():
        os.system("git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git /content/NLP-Lab-works")
    ROOT = Path("/content/NLP-Lab-works")
else:
    p = Path.cwd()
    ROOT = None
    for _ in range(6):
        if (p / "src" / "json_schema.py").exists():
            ROOT = p; break
        p = p.parent
    if ROOT is None:
        raise FileNotFoundError(f"Cannot locate repo root from {Path.cwd()}.")

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"ROOT: {ROOT}")
print("Source modules ready.")


ROOT: /content/NLP-Lab-works
Source modules ready.


## 3. Extraction Task Definition

**Corpus**: 20 Newsgroups — three newsgroup categories about electronics, Christianity, and atheism.

**Task**: From each post fragment, extract 7 structured fields:

| Field | What to extract |
|-------|----------------|
| `category` | Which newsgroup the post belongs to |
| `persons` | Named person mentions |
| `organizations` | Organization / company names |
| `locations` | Location / GPE names |
| `dates` | Date strings verbatim from text |
| `has_question` | Whether the post asks a question |
| `sentiment` | Overall tone of the post |

**Why this task?**
- Builds directly on Lab 10 NER entities (PERSON, ORG, GPE, DATE)
- Adds structured classification fields (category, has_question, sentiment)
- Covers all three corpus domains
- Natural mix of explicit (dates, names) and implicit (sentiment, category) fields

In [21]:
# Show example extraction
example_text = "Intel released its first microprocessor in November 1971. The MIT Media Lab has been doing great work on signal processing."
example_gold = {
    "category": "sci.electronics",
    "persons": [],
    "organizations": ["Intel", "MIT Media Lab"],
    "locations": [],
    "dates": ["November 1971"],
    "has_question": False,
    "sentiment": "positive"
}
print("Example text:")
print(f"  {example_text}")
print()
print("Expected extraction:")
for k, v in example_gold.items():
    print(f"  {k:<18}: {v}")

Example text:
  Intel released its first microprocessor in November 1971. The MIT Media Lab has been doing great work on signal processing.

Expected extraction:
  category          : sci.electronics
  persons           : []
  organizations     : ['Intel', 'MIT Media Lab']
  locations         : []
  dates             : ['November 1971']
  has_question      : False
  sentiment         : positive


## 4. JSON Schema Design

Formal JSON Schema (Draft-07) with type constraints, enum restrictions, and required fields.
All 7 fields are **required**. `additionalProperties: false` prevents extra fields.

In [22]:
from json_schema import get_schema, EXTRACTION_SCHEMA
import json

schema = get_schema()

print("Extraction Schema -- 7 required fields")
print("=" * 38)
for field, prop in schema["properties"].items():
    req = "required" if field in schema["required"] else "optional"
    t = prop["type"]
    extra = ""
    if "enum" in prop:
        extra = f"  enum {prop['enum']}"
    elif t == "array":
        extra = f"  items: {prop['items']['type']}"
    print(f"  {field:<18}: {t:<8} {req}{extra}")

print()
print(f"additionalProperties: {schema.get('additionalProperties', True)}")
print(f"All {len(schema['required'])} fields are REQUIRED.")

Extraction Schema -- 7 required fields
  category          : string   required  enum ['sci.electronics', 'soc.religion.christian', 'alt.atheism']
  persons           : array    required  items: string
  organizations     : array    required  items: string
  locations         : array    required  items: string
  dates             : array    required  items: string
  has_question      : boolean  required
  sentiment         : string   required  enum ['positive', 'negative', 'neutral', 'mixed']

additionalProperties: False
All 7 fields are REQUIRED.


In [23]:
# Print full schema as JSON
print("Full JSON Schema:")
print(json.dumps(EXTRACTION_SCHEMA, indent=2))

Full JSON Schema:
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "NewsGroupExtractionSchema",
  "description": "Structured extraction output for 20 Newsgroups post fragments",
  "type": "object",
  "required": [
    "category",
    "persons",
    "organizations",
    "locations",
    "dates",
    "has_question",
    "sentiment"
  ],
  "properties": {
    "category": {
      "type": "string",
      "enum": [
        "sci.electronics",
        "soc.religion.christian",
        "alt.atheism"
      ],
      "description": "Newsgroup category the text belongs to"
    },
    "persons": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Person names mentioned (empty [] if none)"
    },
    "organizations": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Organization names mentioned (empty [] if none)"
    },
    "locations": {
      "type": "array",
      "items": {
        "

## 5. Evaluation Set

20 texts hand-selected from the 20 Newsgroups corpus:
- 9 from `sci.electronics` — circuit discussions, electronics components
- 6 from `soc.religion.christian` — religious figures, scripture, historical dates
- 5 from `alt.atheism` — philosophers, secular organizations, books

Each text has a full gold annotation for qualitative comparison.

In [24]:
from llm_extract import EVAL_TEXTS, GOLD_SET

cats = {"sci.electronics": 0, "soc.religion.christian": 0, "alt.atheism": 0}
for g in GOLD_SET:
    cats[g["category"]] += 1

print(f"Evaluation set: {len(EVAL_TEXTS)} texts")
for cat, cnt in cats.items():
    print(f"  {cat:<28}: {cnt:2d} texts")

print()
print("Sample gold annotations:")
print()
for i in [0, 9, 15]:
    g = GOLD_SET[i]
    print(f"[{i:02d}] {EVAL_TEXTS[i][:75]}...")
    print(f"     category={g['category']}  persons={g['persons']}  "
          f"has_question={g['has_question']}  sentiment={g['sentiment']}")
    print()

Evaluation set: 20 texts
  sci.electronics             :  9 texts
  soc.religion.christian      :  6 texts
  alt.atheism                 :  5 texts

Sample gold annotations:

[00] I'm using a 2N2222 transistor and a 10k resistor to drive an LED. Can you h...
     category=sci.electronics  persons=[]  has_question=True  sentiment=neutral

[09] Jesus Christ is the Son of God according to Christian belief. Paul wrote to...
     category=soc.religion.christian  persons=['Jesus Christ', 'Paul']  has_question=False  sentiment=positive

[15] Richard Dawkins wrote The God Delusion in 2006. David Hume was an 18th-cent...
     category=alt.atheism  persons=['Richard Dawkins', 'David Hume']  has_question=False  sentiment=neutral



## 6. Baseline Extraction Prompt

The prompt enforces JSON-only output with explicit field descriptions and rules.

Design choices:
1. **Enumerate all fields** with types and examples — reduces hallucination
2. **Explicit null rule** — use `[]` for absent arrays, never `null`
3. **Strict "no extra text" rule** — prevents code fences and prose explanations
4. **Repair prompt** adds: broken output + specific error + type constraints reminder

In [25]:
from llm_extract import build_extraction_prompt, build_repair_prompt, EXTRACTION_PROMPT_TEMPLATE

print("Extraction prompt template:")
print("=" * 60)
# Show the template with placeholder
print(EXTRACTION_PROMPT_TEMPLATE[:600].replace("{text}", "<INPUT TEXT>"))
print("=" * 60)

Extraction prompt template:
You are an information extraction system for 20 Newsgroups posts.
Extract structured information from the text below.

Return ONLY a valid JSON object with EXACTLY these fields:
  "category"      : one of ["sci.electronics", "soc.religion.christian", "alt.atheism"]
  "persons"       : array of person names mentioned (use [] if none)
  "organizations" : array of organization names (use [] if none)
  "locations"     : array of location / place names (use [] if none)
  "dates"         : array of date strings verbatim from text (use [] if none)
  "has_question"  : boolean true if text contains a q


In [26]:
# Show a repair prompt example
broken_example = '{"category": "atheism", "persons": ["Robert Ingersoll"], "organizations": ["American Atheists"], "locations": [], "dates": ["1963"], "has_question": false, "sentiment": "neutral"}'
error_example  = "'atheism' is not one of ['sci.electronics', 'soc.religion.christian', 'alt.atheism']"
repair_p = build_repair_prompt(EVAL_TEXTS[16], broken_example, error_example)
print("Repair prompt (for enum violation case):")
print("-" * 60)
print(repair_p[:700])
print("-" * 60)

Repair prompt (for enum violation case):
------------------------------------------------------------
The previous extraction attempt returned an invalid output.

Original text:
The American Atheists organization was founded in 1963. Robert Ingersoll was a famous 19th-century agnostic.

Broken output:
{"category": "atheism", "persons": ["Robert Ingersoll"], "organizations": ["American Atheists"], "locations": [], "dates": ["1963"], "has_question": false, "sentiment": "neutral"}

Validation error:
'atheism' is not one of ['sci.electronics', 'soc.religion.christian', 'alt.atheism']

Please return a CORRECTED, valid JSON object that:
1. Fixes the specific validation error listed above
2. Contains ONLY these fields: category, persons, organizations, locations, dates, has_question, sentiment
3. U
------------------------------------------------------------


## 7. Raw Extraction

Run MockLLM on all 20 evaluation texts (attempt 0 — no repair).

The MockLLM simulates realistic LLM behaviour with 6 intentional failure modes:
- 2 parse errors (code fence, trailing text)
- 1 permanent parse error (not JSON)
- 1 missing required field
- 1 wrong type (boolean as string)
- 1 enum violation

In production, replace `MockLLM` with an actual LLM call (Gemini, OpenAI, HF Inference).

In [27]:
from llm_extract import MockLLM, EVAL_TEXTS, build_extraction_prompt

llm = MockLLM(noise_level="low")

raw_responses = []
for i, text in enumerate(EVAL_TEXTS):
    prompt = build_extraction_prompt(text)
    resp = llm.call(prompt, text_idx=i, attempt=0)
    raw_responses.append(resp)

print(f"Raw extraction complete: {len(raw_responses)} responses collected.")
print()
print("Sample raw responses:")
for i in [0, 3, 6, 8, 14, 16]:
    print(f"  [{i:02d}] {raw_responses[i][:90].replace(chr(10),'|')!r}{'...' if len(raw_responses[i]) > 90 else ''}")

Raw extraction complete: 20 responses collected.

Sample raw responses:
  [00] '{"category": "sci.electronics", "persons": [], "organizations": [], "locations": [], "date'...
  [03] '```json|{"category": "sci.electronics", "persons": [], "organizations": [], "locations": ['...
  [06] '{"category": "sci.electronics", "persons": [], "organizations": ["Farnell Electronics"], "'...
  [08] '{"category": "sci.electronics", "persons": [], "organizations": ["Cleveland State Universi'...
  [14] 'The extracted information from the text is as follows: The post category is soc.religion.c'...
  [16] '{"category": "atheism", "persons": ["Robert Ingersoll"], "organizations": ["American Athei'...


In [28]:
from validator import validate_batch

raw_results = validate_batch(raw_responses)

print("Raw LLM extraction results (20 texts):")
print("-" * 60)
for i, r in enumerate(raw_results):
    cat = ""
    if r.parse_ok and r.parsed:
        cat = f" | category={r.parsed.get('category','?')}"
    status = "VALID  " if r.valid else "INVALID"
    err = "" if r.valid else f" | {r.short_error()}"
    print(f"[{i:02d}] parse={'OK  ' if r.parse_ok else 'FAIL'} "
          f"schema={'OK  ' if r.schema_ok else 'FAIL'} "
          f"schema={'N/A ' if not r.parse_ok else ('OK  ' if r.schema_ok else 'FAIL')} "
          f"| {status}{cat}{err}")

Raw LLM extraction results (20 texts):
------------------------------------------------------------
[00] parse=OK   schema=OK   schema=OK   | VALID   | category=sci.electronics
[01] parse=OK   schema=OK   schema=OK   | VALID   | category=sci.electronics
[02] parse=OK   schema=OK   schema=OK   | VALID   | category=sci.electronics
[03] parse=FAIL schema=FAIL schema=N/A  | INVALID | parse_error: code fence wrapping
[04] parse=OK   schema=OK   schema=OK   | VALID   | category=sci.electronics
[05] parse=OK   schema=OK   schema=OK   | VALID   | category=sci.electronics
[06] parse=FAIL schema=FAIL schema=N/A  | INVALID | parse_error: trailing text / malformed (Extra data: line 3 column 1 (char 176))
[07] parse=OK   schema=OK   schema=OK   | VALID   | category=sci.electronics
[08] parse=OK   schema=FAIL schema=FAIL | INVALID | category=sci.electronics | schema_violation: 'sentiment' is a required property
[09] parse=OK   schema=OK   schema=OK   | VALID   | category=soc.religion.christian
[10] 

## 8. JSON Validator

The validator performs two independent checks:
1. **Parse check** — `json.loads()` — catches syntax errors, code fences, trailing text
2. **Schema check** — `jsonschema.validate()` — catches type errors, missing fields, enum violations

These are reported separately because they have different root causes and different repair strategies.

In [29]:
from validator import validation_summary

raw_summary = validation_summary(raw_results)
print("Validation summary (raw extraction):")
print(f"  Total         : {raw_summary['total']}")
print(f"  Parse OK      : {raw_summary['parse_ok']}")
print(f"  Parse FAIL    : {raw_summary['parse_fail']}")
print(f"  Schema OK     : {raw_summary['schema_ok']}")
print(f"  Schema FAIL   : {raw_summary['schema_fail']}")
print(f"  VALID (both)  : {raw_summary['valid']}  ({raw_summary['valid_rate']*100:.1f}%)")
print(f"  INVALID       : {raw_summary['invalid']}")

Validation summary (raw extraction):
  Total         : 20
  Parse OK      : 17
  Parse FAIL    : 3
  Schema OK     : 14
  Schema FAIL   : 3
  VALID (both)  : 14  (70.0%)
  INVALID       : 6


In [30]:
# Detailed breakdown of the 6 failures
failures = [(i, r) for i, r in enumerate(raw_results) if not r.valid]
print(f"Validator detail -- {len(failures)} failed cases:")
print("=" * 60)
for i, r in failures:
    etype = r.error_type()
    print(f"[{i:02d}] ERROR TYPE : {etype}")
    raw_preview = raw_responses[i][:80].replace('\n', '|')
    print(f"     RAW OUTPUT : {raw_preview!r}{'...' if len(raw_responses[i])>80 else ''}")
    if etype == "parse_error":
        print(f"     DIAGNOSIS  : {r.short_error()}")
        print(f"     json.loads : {r.parse_error[:80]}")
    else:
        print(f"     DIAGNOSIS  : {r.first_schema_error()}")
    print()

Validator detail -- 6 failed cases:
[03] ERROR TYPE : parse_error
     RAW OUTPUT : '```json|{"category": "sci.electronics", "persons": [], "organizations": [], "loc'...
     DIAGNOSIS  : parse_error: code fence wrapping
     json.loads : Expecting value: line 1 column 1 (char 0)

[06] ERROR TYPE : parse_error
     RAW OUTPUT : '{"category": "sci.electronics", "persons": [], "organizations": ["Farnell Electr'...
     DIAGNOSIS  : parse_error: trailing text / malformed (Extra data: line 3 column 1 (char 176))
     json.loads : Extra data: line 3 column 1 (char 176)

[08] ERROR TYPE : schema_violation
     RAW OUTPUT : '{"category": "sci.electronics", "persons": [], "organizations": ["Cleveland Stat'...
     DIAGNOSIS  : 'sentiment' is a required property

[12] ERROR TYPE : schema_violation
     RAW OUTPUT : '{"category": "soc.religion.christian", "persons": ["Mother Teresa"], "organizati'...
     DIAGNOSIS  : 'false' is not of type 'boolean'

[14] ERROR TYPE : parse_error
     RAW OUTPU

## 9. Repair Loop

For each failed extraction, we build a **repair prompt** that includes:
1. The original text
2. The broken LLM output
3. The specific validation error message
4. Instructions to fix exactly that error

Maximum 1 repair attempt per text (max 2 total LLM calls per text).

The repair loop is implemented in `src/repair_loop.py`.

In [31]:
from repair_loop import run_pipeline, pipeline_metrics, print_pipeline_metrics

pipeline_results = run_pipeline(llm, EVAL_TEXTS)

# Show repair actions
repair_needed = [r for r in pipeline_results if r.needed_repair]
print(f"Running repair loop on {len(repair_needed)} failed examples...")
print("=" * 60)
for r in repair_needed:
    err = r.raw_valid.short_error()
    fixed = "FIXED" if r.repair_helped else "STILL INVALID (permanent failure)"
    repair_preview = (r.repair_response or "")[:80].replace('\n', ' ')
    print(f"[{r.text_idx:02d}] Attempt 1 repair:")
    print(f"     Error      : {err}")
    print(f"     Repair resp: {repair_preview!r}{'...' if len(r.repair_response or '')>80 else ''}")
    print(f"     Result     : parse={'OK' if r.final_valid.parse_ok else 'FAIL'}"
          f"  schema={'OK' if r.final_valid.schema_ok else 'FAIL'}"
          f"  --> {fixed}")
    print()

print("=" * 60)
fixed_count = sum(1 for r in repair_needed if r.repair_helped)
print(f"Repair summary: {fixed_count} fixed / {len(repair_needed)} attempted = {fixed_count/len(repair_needed)*100:.1f}%")

Running repair loop on 6 failed examples...
[03] Attempt 1 repair:
     Error      : parse_error: code fence wrapping
     Repair resp: '{"category": "sci.electronics", "persons": [], "organizations": [], "locations":'...
     Result     : parse=OK  schema=OK  --> FIXED

[06] Attempt 1 repair:
     Error      : parse_error: trailing text / malformed (Extra data: line 3 column 1 (char 176))
     Repair resp: '{"category": "sci.electronics", "persons": [], "organizations": ["Farnell Electr'...
     Result     : parse=OK  schema=OK  --> FIXED

[08] Attempt 1 repair:
     Error      : schema_violation: 'sentiment' is a required property
     Repair resp: '{"category": "sci.electronics", "persons": [], "organizations": ["Cleveland Stat'...
     Result     : parse=OK  schema=OK  --> FIXED

[12] Attempt 1 repair:
     Error      : schema_violation: 'false' is not of type 'boolean'
     Repair resp: '{"category": "soc.religion.christian", "persons": ["Mother Teresa"], "organizati'...
     Resu

## 10. Metrics: Valid JSON Rate

The main metric for this lab is **valid JSON rate** — the fraction of examples
that produce a valid, schema-conformant JSON object.

Three variants required by the lab:
1. **Raw valid JSON rate** — before repair loop
2. **Post-repair valid JSON rate** — after repair loop
3. **Schema-valid JSON rate** — specifically pass schema validation (same as post-repair here)

In [32]:
metrics = pipeline_metrics(pipeline_results)
print_pipeline_metrics(metrics, "Extraction Pipeline Metrics")

# Comparison table
print("\nComparison table:")
raw_rate    = metrics["raw_valid_rate"] * 100
repair_rate = metrics["post_repair_valid_rate"] * 100
raw_parse   = sum(1 for r in raw_results if not r.parse_ok)
rep_parse   = sum(1 for r in pipeline_results if not r.final_valid.parse_ok)
raw_schema  = sum(1 for r in raw_results if r.parse_ok and not r.schema_ok)
rep_schema  = sum(1 for r in pipeline_results if r.final_valid.parse_ok and not r.final_valid.schema_ok)

print(f"{'Metric':<30}  {'Before repair':<16}  {'After repair'}")
print("-" * 66)
n = len(pipeline_results)
print(f"{'Valid JSON rate':<30}  {metrics['raw_valid']}/20 = {raw_rate:5.1f}%   {metrics['post_repair_valid']}/20 = {repair_rate:5.1f}%")
print(f"{'Parse failures':<30}  {raw_parse}/20 = {raw_parse/n*100:5.1f}%   {rep_parse}/20 = {rep_parse/n*100:5.1f}%")
print(f"{'Schema violations':<30}  {raw_schema}/20 = {raw_schema/n*100:5.1f}%   {rep_schema}/20 = {rep_schema/n*100:5.1f}%")
print(f"{'Permanently invalid':<30}  {n-metrics['raw_valid']}/20 = {(n-metrics['raw_valid'])/n*100:5.1f}%   {metrics['final_invalid']}/20 = {metrics['final_invalid']/n*100:5.1f}%")


  Extraction Pipeline Metrics
  Total examples              : 20
--------------------------------------------------
  Raw valid JSON rate         : 14 / 20  = 70.0%
--------------------------------------------------
  Needed repair               :  6 / 20  = 30.0%
  Repair fixed                :  5 / 6   = 83.3%
--------------------------------------------------
  Post-repair valid JSON rate : 19 / 20  = 95.0%
  Improvement                 : +5 examples  +25.0pp
--------------------------------------------------
  Avg LLM calls per example   : 1.30
  % examples repair helped    : 30.0%
  % examples repair failed    : 5.0%


Comparison table:
Metric                          Before repair     After repair
------------------------------------------------------------------
Valid JSON rate                 14/20 =  70.0%   19/20 =  95.0%
Parse failures                  3/20 =  15.0%   1/20 =   5.0%
Schema violations               3/20 =  15.0%   0/20 =   0.0%
Permanently invalid            

## 11. Error Analysis

Structured breakdown of **16 problematic cases**:
- 6 structural/format errors (detected by validator)
- 10 semantic errors (valid JSON but wrong extraction content)

Error categories used:
- **parse_error** — output is not valid JSON
- **schema_violation** — JSON parsed but fails schema
- **semantic_error** — JSON valid but extraction is wrong/incomplete

In [33]:
print("=== Error Analysis: 16 problematic cases ===")
print()

# 6 structural errors
print("--- Structural / Format Errors (6) ---")
print()
fmt = " {:>2} | {:>3} | {:<22} | {:<54} | {}"
header = fmt.format("#", "idx", "error_type", "description", "repaired")
print(header)

structural_errors = [
    (3,  "parse_error",      "code fence: ```json...``` wrapping",              "YES"),
    (6,  "parse_error",      "trailing explanatory text after closing brace",   "YES"),
    (8,  "schema_violation", "missing required field: 'sentiment'",             "YES"),
    (12, "schema_violation", "wrong type: has_question='false' (str not bool)", "YES"),
    (14, "parse_error",      "not JSON at all -- LLM returned prose summary",   "NO (permanent)"),
    (16, "schema_violation", "enum violation: category='atheism' not valid",    "YES"),
]
for n, (idx, etype, desc, rep) in enumerate(structural_errors, 1):
    print(fmt.format(n, idx, etype, desc, rep))

print()
print("--- Semantic Errors in Valid Outputs (10) ---")
print()
semantic_errors = [
    (0,  "semantic_error", "2N2222 transistor/resistor not captured -- no tech_terms field", "N/A"),
    (1,  "semantic_error", "sentiment='positive' for factual statement -- debatable",        "N/A"),
    (5,  "semantic_error", "HP 34401A model number not captured -- no product field",        "N/A"),
    (7,  "semantic_error", "MOSFET/BJT not extracted -- no tech_terms field in schema",      "N/A"),
    (9,  "semantic_error", "'God' not in persons[] -- generic word, arguably correct",       "N/A"),
    (10, "semantic_error", "'Bible' missing -- no document/works field in schema",           "N/A"),
    (11, "semantic_error", "'Pentecost' in dates[] is religious festival, not a date",      "N/A"),
    (14, "semantic_error", "persons/dates correctly identified but output invalid",          "NO"),
    (15, "semantic_error", "'The God Delusion' not captured -- no works field",              "N/A"),
    (17, "semantic_error", "Voyager mission not captured -- no events field in schema",      "N/A"),
]
for n, (idx, etype, desc, rep) in enumerate(semantic_errors, 1):
    print(fmt.format(n, idx, etype, desc, rep))

=== Error Analysis: 16 problematic cases ===

--- Structural / Format Errors (6) ---

  # | idx | error_type             | description                                            | repaired
  1 |   3 | parse_error            | code fence: ```json...``` wrapping                     | YES
  2 |   6 | parse_error            | trailing explanatory text after closing brace          | YES
  3 |   8 | schema_violation       | missing required field: 'sentiment'                    | YES
  4 |  12 | schema_violation       | wrong type: has_question='false' (str not bool)        | YES
  5 |  14 | parse_error            | not JSON at all -- LLM returned prose summary          | NO (permanent)
  6 |  16 | schema_violation       | enum violation: category='atheism' not valid           | YES

--- Semantic Errors in Valid Outputs (10) ---

  1 |   0 | semantic_error         | 2N2222 transistor/resistor not captured -- no tech_terms field | N/A
  2 |   1 | semantic_error         | sentiment='positive' 

### Error Analysis Summary

**Most frequent category: `semantic_error` (10/16)**
Root cause: Schema does not have fields for technical components, product names,
religious events, or document titles. These information types are simply lost.

**Structural errors: equally split (3 parse_error + 3 schema_violation)**
All structural errors except one were fixed by the repair loop.

**What repair loop covers well:**
- Code fence wrapping (strip and retry)
- Trailing text (LLM removes on repair)
- Missing fields (LLM adds when told exactly which field)
- Wrong types (LLM fixes when told the expected type explicitly)
- Enum violations (LLM picks correct value when reminded of options)

**What repair loop cannot fix:**
- LLM that consistently returns prose summaries (permanent hallucination)
- Semantic errors (JSON is valid but content is wrong — outside schema scope)
- Schema gaps (fields that don't exist in the schema — design issue)

**Key finding**: Schema-first pipeline turns 70% → 95% valid JSON rate with a
single repair attempt. The remaining 5% failure is a fundamental LLM behaviour
issue (prose mode) that cannot be resolved at the prompt level alone.

## 12. Generate `docs/audit_summary_lab11.md`

In [34]:
from pathlib import Path

DOCS_DIR = ROOT / "docs"
DOCS_DIR.mkdir(exist_ok=True)

audit_content = """# Audit Summary -- Lab 11: LLM Extraction (schema-first)

**Date:** 2026-05-29

## 1. Extraction Case
Task: Structured extraction from 20 Newsgroups post fragments
Corpus: 20 Newsgroups -- alt.atheism / sci.electronics / soc.religion.christian
Schema fields (7): category | persons | organizations | locations | dates | has_question | sentiment

## 2. Evaluation Set
Total texts: 20
sci.electronics: 9 | soc.religion.christian: 6 | alt.atheism: 5

## 3. Raw Valid JSON Rate
14 / 20 = 70.0%
Parse failures: 3/20 (code fence, trailing text, not JSON)
Schema violations: 3/20 (missing field, wrong type, enum violation)

## 4. Post-Repair Valid JSON Rate
19 / 20 = 95.0%
Repair needed: 6/20 | Repair fixed: 5/6 (83.3%) | Permanently invalid: 1/20

## 5. Schema-Valid JSON Rate
19 / 20 = 95.0%

## 6. Most Problematic Fields
has_question: string instead of boolean
category: incorrect enum value
sentiment: field omitted by LLM
dates: religious calendar terms treated as dates

## 7. Error Types
parse_error: 3 | schema_violation: 3 | semantic_error: 10

## 8. Schema-first Pipeline Assessment
Repair loop: +25pp improvement (70 -> 95%). Semantic errors not detectable by schema alone.
"""

audit_path = DOCS_DIR / "audit_summary_lab11.md"
audit_path.write_text(audit_content, encoding="utf-8")
print(f"Saved: {audit_path}")

Saved: /content/NLP-Lab-works/docs/audit_summary_lab11.md
